# Attention Pattern Complexity

Complexity metrics for attention patterns: entropy, effective rank,
regularity (self/prev/first dominance), and positional stability.

## Why This Matters

Attention pattern complexity reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_complexity import (
    pattern_entropy_complexity, pattern_rank_complexity,
    pattern_regularity, pattern_stability_across_positions,
    pattern_complexity_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Entropy Complexity

High entropy = complex/diffuse; low = simple/focused.

In [ ]:
for layer in range(4):
    result = pattern_entropy_complexity(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_complex_heads']} complex heads")
    for h in result['per_head']:
        flag = ' [COMPLEX]' if h['is_complex'] else ''
        print(f"    H{h['head']}: H={h['mean_entropy']:.4f}, norm_H={h['normalized_entropy']:.4f}{flag}")

## Rank Complexity

Effective rank of the attention pattern matrix.

In [ ]:
for layer in range(4):
    result = pattern_rank_complexity(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_rank={result['mean_rank']:.2f}")
    for h in result['per_head']:
        print(f"    H{h['head']}: rank={h['effective_rank']:.2f}, top_sv={h['top_sv']:.4f}")

## Pattern Regularity

Self-attention, previous-token, or first-token dominance?

In [ ]:
for layer in range(4):
    for head in range(4):
        r = pattern_regularity(model, tokens, layer, head)
        print(f"  L{layer}H{head}: {r['dominant_pattern']:5s} | "
              f"self={r['self_attention']:.4f}, prev={r['prev_token_attention']:.4f}, "
              f"first={r['first_token_attention']:.4f}")

## Pattern Stability

How stable are patterns across query positions?

In [ ]:
for layer in range(4):
    result = pattern_stability_across_positions(model, tokens, layer=layer)
    print(f"  Layer {layer}: stability={result['mean_stability']:.4f}")
    for h in result['per_head']:
        print(f"    H{h['head']}: stability={h['stability']:.4f}")

## Complexity Summary

In [ ]:
result = pattern_complexity_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: complex_heads={p['n_complex_heads']}, "
          f"mean_rank={p['mean_rank']:.2f}")